<a href="https://colab.research.google.com/github/bhardwaj-amit79/Colab/blob/main/Prefix_Tuning_Demo_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### <font color="blue"><b>Prefix Tuning (Li & Liang., ACL Findings 2021) </b></font>

Session Objectives - By the end of this code demo, participants will be able to:

1. Understand the concept of Prefix-Tuning for efficient model adaptation.
2. Prepare data for text summarization tasks.
3. Understand Prefix-Tuning, Full Fine-Tuning, and Frozen Model approaches.
4. Evaluate summarization performance using ROUGE metrics.
5. Explain prefix-only training with a frozen backbone model.
6. Understand the full fine-tuning process.
7. Compare different fine-tuning approaches.
8. Understand the training and validation process.
9. Interpret loss curves and parameter statistics to evaluate model performance.

In [ ]:
!pip install -q rouge_score

#### <font color="maroon"><b>Essential Packages/Libraries Imports</b></font>

In [ ]:
import math
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from transformers import BartForConditionalGeneration, BartTokenizer, get_linear_schedule_with_warmup
from peft import get_peft_model, PrefixTuningConfig, TaskType
from datasets import load_dataset
from rouge_score import rouge_scorer

#### <font color="maroon"><b>Function to set seeds for Reproducibility & Configuration </b></font>

In [ ]:
def set_seed(seed=42):
    random.seed(seed)                             # Python random
    np.random.seed(seed)                          # Numpy's random
    torch.manual_seed(seed)                       # torch's random
    torch.cuda.manual_seed_all(seed)              # cuda's random
    torch.backends.cudnn.deterministic = True     # deterministic cuDNN

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME    = "facebook/bart-large"
TRAIN_SAMPLES = 200        # training samples      (paper: 204K)
TEST_SAMPLES  = 20         # test samples          (200+ recommended)
BATCH_SIZE    = 4          # paper: 14 on a V100
GRAD_ACCUM    = 4
MAX_SRC_LEN   = 512        # paper truncates articles to 512 BPE
MAX_TGT_LEN   = 64
LEARNING_RATE = 1e-4       # prefix-tuning (small param set tolerates higher LR)
LR_FULL_FT    = 1e-5       # BART-large full FT diverges at higher LRs
EPOCHS        = 20         # ceiling; early stopping decides the actual count
PATIENCE      = 3          # epochs without val improvement before stopping
PREFIX_VALUES = [50, 100, 200]        # |P_idx|               (paper's XSUM setting)
BOTTLENECK    = 800        # k in P'_theta         (paper's k for summarization)
RUN_FULL_FT   = True       # needs ~2x the VRAM of the prefix run

# BART-large overflows in fp16 during full fine-tuning; force fp32 everywhere.
DTYPE = torch.float32

#### <font color="maroon"><b> Dataset Loading & Collate Function </b></font>

In [ ]:
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
xsum_train_raw = load_dataset("EdinburghNLP/xsum", split=f"train[:{TRAIN_SAMPLES}]")
xsum_test_raw  = load_dataset("EdinburghNLP/xsum", split=f"validation[:{TEST_SAMPLES}]")

In [ ]:
print(xsum_train_raw[0])
print("---------------")
print(xsum_train_raw[0]['document'])
print("---------------")
print(xsum_train_raw[0]['summary'])

In [ ]:
def collate_fn(batch, tok):
    src = tok([b["document"] for b in batch], max_length=MAX_SRC_LEN, truncation=True, padding=True, return_tensors="pt")
    tgt = tok([b["summary"] for b in batch], max_length=MAX_TGT_LEN, truncation=True, padding=True, return_tensors="pt")
    labels = tgt.input_ids.masked_fill(tgt.input_ids == tok.pad_token_id, -100)
    return {"input_ids": src.input_ids,
            "attention_mask": src.attention_mask,
            "labels": labels}

train_loader = DataLoader(xsum_train_raw, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda b: collate_fn(b, tokenizer))
val_loader   = DataLoader(xsum_test_raw,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_fn(b, tokenizer))

#### <font color="maroon"><b> Metric calculation - Summarization </b></font>

In [ ]:
def compute_rouge(preds, golds) -> dict:
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'],
                                      use_stemmer=True)
    agg = {"ROUGE-1": 0.0, "ROUGE-2": 0.0, "ROUGE-L": 0.0}
    for p, g in zip(preds, golds):
        s = scorer.score(g, p)
        agg["ROUGE-1"] += s['rouge1'].fmeasure * 100
        agg["ROUGE-2"] += s['rouge2'].fmeasure * 100
        agg["ROUGE-L"] += s['rougeL'].fmeasure * 100
    n = len(preds)
    return {k: v / n for k, v in agg.items()}

#### <font color="maroon"><b> Function to generate predictions </b></font>

In [ ]:
@torch.no_grad()
def generate_predictions(model, dataset, num_beams=6, max_gen_len=62,
                         length_penalty=0.8, batch_size=8):
    model.eval()
    preds = []
    for i in range(0, len(dataset), batch_size):
        chunk = dataset[i:i + batch_size]
        x = tokenizer(chunk["document"], max_length=MAX_SRC_LEN, truncation=True,
                      padding=True, return_tensors="pt").to(DEVICE)
        out = model.generate(input_ids=x.input_ids,
                             attention_mask=x.attention_mask,
                             num_beams=num_beams, max_length=max_gen_len,
                             min_length=11, length_penalty=length_penalty,
                             no_repeat_ngram_size=3, early_stopping=True)
        preds += tokenizer.batch_decode(out, skip_special_tokens=True)
    model.train()
    return preds

#### <font color="maroon"><b> Loss curves </b></font>

In [ ]:
@torch.no_grad()
def evaluate_loss(model, loader):
    """Token-weighted mean cross-entropy (HF loss is already token-mean)."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        n_tok = (batch["labels"] != -100).sum().item()
        if n_tok == 0:
            continue
        outputs = model(**batch)
        total_loss += outputs.loss.item() * n_tok
        total_tokens += n_tok
    model.train()
    return total_loss / max(1, total_tokens)


def plot_loss_curves(history: dict, out_path="loss_curves.png"):
    arms = list(history.keys())
    fig, axes = plt.subplots(1, len(arms), figsize=(6 * len(arms), 5), squeeze=False)

    for ax, arm_name in zip(axes[0], arms):
        train = history[arm_name]["train"]
        val = history[arm_name]["val"]
        epochs = range(1, len(train) + 1)

        ax.plot(epochs, train, marker="o", linewidth=2, label="train")
        ax.plot(epochs, val, marker="s", linewidth=2, label="val")
        ax.axhline(history[arm_name]["v0"], ls="--", c="gray", lw=1,
                   label="untrained val")
        ax.axvline(history[arm_name]["best_epoch"], ls=":", c="green", lw=1.5,
                   label=f"best epoch ({history[arm_name]['best_epoch']})")

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title(arm_name)
        ax.set_xticks(list(epochs))
        ax.grid(True, alpha=0.3)
        ax.legend()

    fig.suptitle("Prefix-Tuning vs Full Fine-Tuning — Train vs. Validation Loss")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)
    print(f"\nSaved loss curves -> {out_path}")

#### <font color="maroon"><b> Training Loop </b></font>

In [ ]:
def train_arm(model, lr, arm_name):

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    total_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps
    )

    print(f"  [{arm_name}] dtype: {next(model.parameters()).dtype} | "
          f"trainable tensors: {len(params)}")

    v0 = evaluate_loss(model, val_loader)
    print(f"  [{arm_name}] Untrained | Val Loss: {v0:.4f}")

    train_losses, val_losses = [], []
    best_val, best_state, best_epoch, bad = float("inf"), None, 0, 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, total_tokens = 0.0, 0

        for i, batch in enumerate(train_loader):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            n_tok = (batch["labels"] != -100).sum().item()
            if n_tok == 0:                       # all targets truncated away
                continue

            outputs = model(**batch)

            if not torch.isfinite(outputs.loss):
                raise RuntimeError(
                    f"[{arm_name}] non-finite loss at epoch {epoch}, batch {i}. "
                    f"dtype={next(model.parameters()).dtype}, lr={lr}")

            (outputs.loss / GRAD_ACCUM).backward()
            total_loss += outputs.loss.item() * n_tok
            total_tokens += n_tok

            if (i + 1) % GRAD_ACCUM == 0:
                gnorm = torch.nn.utils.clip_grad_norm_(params, 1.0)
                if not torch.isfinite(gnorm):
                    raise RuntimeError(
                        f"[{arm_name}] non-finite gradient norm at epoch {epoch}, "
                        f"batch {i}. Lower the learning rate.")
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        avg_loss = total_loss / max(1, total_tokens)
        val_loss = evaluate_loss(model, val_loader)
        train_losses.append(avg_loss)
        val_losses.append(val_loss)
        print(f"  [{arm_name}] Epoch {epoch}/{EPOCHS} | "
              f"Train Loss: {avg_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val:
            best_val, best_epoch, bad = val_loss, epoch, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"  [{arm_name}] Early stop at epoch {epoch} "
                      f"(no improvement for {PATIENCE} epochs)")
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
        del best_state
        torch.cuda.empty_cache()

    print(f"  [{arm_name}] Best Val Loss: {best_val:.4f} at epoch {best_epoch} "
          f"(restored for evaluation)")
    return {"train": train_losses, "val": val_losses,
            "v0": v0, "best_epoch": best_epoch}

## <font color="maroon"><b> Main Piepline Execution </b></font>

In [ ]:
loss_history = {}     # arm_name -> {"train": [...], "val": [...], "v0", "best_epoch"}
results = {}          # arm_name -> {"ROUGE-1": ..., ...}
samples = {}          # arm_name -> first prediction

# ---- Arm 1: no fine-tuning (raw pretrained BART-large) ----
print("=" * 65)
print(" Arm: 'no fine-tuning (raw BART)'")
print("=" * 65)

base = BartForConditionalGeneration.from_pretrained(MODEL_NAME, torch_dtype=DTYPE).to(DEVICE)
preds = generate_predictions(base, xsum_test_raw)
golds = list(xsum_test_raw["summary"])
results["no fine-tuning"] = compute_rouge(preds, golds)
samples["no fine-tuning"] = preds[0]
r = results["no fine-tuning"]
print(f"  [no fine-tuning] TEST EVALUATION | ROUGE-1: {r['ROUGE-1']:.2f} | "
      f"ROUGE-2: {r['ROUGE-2']:.2f} | ROUGE-L: {r['ROUGE-L']:.2f}\n")

### <font color="maroon">**Prefix-Tuning Illustration**</font>


Link - https://huggingface.co/docs/peft/package_reference/prefix_tuning

<img src="https://raw.githubusercontent.com/Rekha215/Demo-Images/refs/heads/main/prefix_image.png" width="600">

| PEFT Config Parameter | Corresponding Part in the Image |
|------------------------|---------------------------------|
| `num_virtual_tokens = 20` | The **orange blocks** at the beginning (learnable virtual prefix tokens). |
| `token_dim = 768` | The dimension of each orange block before entering the PrefixEncoder MLP. In the figure, this is \(P'_\theta[i,:] \in \mathbb{R}^{768}\). |
| `encoder_hidden_size = 768` | The hidden layer size of the **PrefixEncoder MLP**, represented by the yellow trapezoid labeled \(\theta\). |
| `num_layers = 12` | The generated prefix is replicated and supplied to all **12 Transformer layers** (not shown explicitly in the figure). |
| `num_attention_heads = 12` | The MLP output is reshaped into **12 attention heads** to form the prefix key/value vectors. |
| `num_transformer_submodules = 1` | One set of prefix representations is generated for a single Transformer submodule (e.g., encoder or decoder, depending on the model). |

----
| `prefix_projection` | Representation in the Image |
|---------------------|-----------------------------|
| `True` | **Matches the image.** The yellow **MLP (\(\theta\))** represents the PrefixEncoder (projection layer), where \(P_\theta[i,:] = \mathrm{MLP}_\theta(P'_\theta[i,:])\). |
| `False` | The **MLP is omitted**. The learnable virtual prefix tokens are used directly as the prefix representations without passing through a projection layer. |
-----

<font color="red">**Note: The important point is that prefix_projection=True only affects how P_k and P_v are generated. The rest of the table (queries, keys, values, attention computation) is part of Prefix Tuning itself, regardless of whether projection is enabled.**</font>

<img src="https://raw.githubusercontent.com/Rekha215/Demo-Images/refs/heads/main/kv_image.png" width="600">


| Component / Equation | Corresponding Part in the Image / Your Code |
|----------------------|---------------------------------------------|
| `Q` (Query) | Computed from the original hidden states. It is **not modified** by Prefix Tuning. |
| `K` (Key) | Original key vectors computed by the BART attention layers. |
| `V` (Value) | Original value vectors computed by the BART attention layers. |
| `P_k` | Prefix key representations generated from the learnable virtual tokens by the PrefixEncoder (`prefix_projection=True`). |
| `P_v` | Prefix value representations generated from the learnable virtual tokens by the PrefixEncoder (`prefix_projection=True`). |
| `K' = [P_k; K]` | Conceptually, the attention uses the prefix keys together with the original keys. In PEFT, this is implemented by supplying **past key tensors** rather than explicitly concatenating matrices. |
| `V' = [P_v; V]` | Conceptually, the attention uses the prefix values together with the original values. In PEFT, this is implemented by supplying **past value tensors** rather than explicitly concatenating matrices. |
| `Attn(Q,K,V)` | Standard BART self-attention without Prefix Tuning. |
| `Attn(Q,K',V')` | Prefix Tuning attention using the generated prefix key/value representations. |
| `;` (semicolon) | Conceptually denotes concatenation of the prefix and original key/value vectors. |

In [ ]:
for N_PREFIX in PREFIX_VALUES:

    print("=" * 65)
    print(f" Arm: 'prefix-tuning' (N_PREFIX={N_PREFIX})")
    print("=" * 65)

    base = BartForConditionalGeneration.from_pretrained(
        MODEL_NAME, torch_dtype=DTYPE
    )

    peft_config = PrefixTuningConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        num_virtual_tokens=N_PREFIX,
        encoder_hidden_size=BOTTLENECK,
        prefix_projection=True,
    )

    prefix_model = get_peft_model(base, peft_config).to(DEVICE)

    arm_name = f"prefix-{N_PREFIX}"

    loss_history[arm_name] = train_arm(
        prefix_model, LEARNING_RATE, arm_name
    )

    preds = generate_predictions(prefix_model, xsum_test_raw)
    results[arm_name] = compute_rouge(preds, golds)
    samples[arm_name] = preds[0]

    r = results[arm_name]
    print(
        f"[{arm_name}] "
        f"ROUGE-1: {r['ROUGE-1']:.2f} | "
        f"ROUGE-2: {r['ROUGE-2']:.2f} | "
        f"ROUGE-L: {r['ROUGE-L']:.2f}"
    )

    prefix_model.cpu()
    torch.cuda.empty_cache()

In [ ]:
# ---- Arm 3: full fine-tuning ----
if RUN_FULL_FT:
    print("=" * 65)
    print(" Arm: 'full fine-tuning'")
    print("=" * 65)

    prefix_model = prefix_model.cpu()
    torch.cuda.empty_cache()

    ft_model = BartForConditionalGeneration.from_pretrained(
        MODEL_NAME, torch_dtype=DTYPE).to(DEVICE)
    loss_history["full fine-tuning"] = train_arm(ft_model, LR_FULL_FT,
                                                 "full fine-tuning")
    preds = generate_predictions(ft_model, xsum_test_raw)
    results["full fine-tuning"] = compute_rouge(preds, golds)
    samples["full fine-tuning"] = preds[0]
    r = results["full fine-tuning"]
    print(f"  [full fine-tuning] TEST EVALUATION | ROUGE-1: {r['ROUGE-1']:.2f} | "
          f"ROUGE-2: {r['ROUGE-2']:.2f} | ROUGE-L: {r['ROUGE-L']:.2f}\n")

    ft_model = ft_model.cpu()
    torch.cuda.empty_cache()
    prefix_model = prefix_model.to(DEVICE)

#### <font color="maroon"><b> Plots </b></font>

In [ ]:
print("=" * 65)
print(" SUMMARY")
print("=" * 65)
print(f"{'':<26}{'ROUGE-1':>9}{'ROUGE-2':>9}{'ROUGE-L':>9}{'Best Ep':>9}")
for name, s in results.items():
    ep = loss_history.get(name, {}).get("best_epoch", "-")
    print(f"{name:<26}{s['ROUGE-1']:9.2f}{s['ROUGE-2']:9.2f}"
          f"{s['ROUGE-L']:9.2f}{str(ep):>9}")

print("\nSample predictions:")
for name, p in samples.items():
    print(f"  [{name}]: '{p}'")
print(f"  [gold]: '{golds[0]}'")

plot_loss_curves(loss_history, out_path="loss_curves.png")

#### <font color="maroon"><b> Parameters </b></font>

In [ ]:
trainable = sum(p.numel() for p in prefix_model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in prefix_model.parameters() if not p.requires_grad)

state = prefix_model.prompt_encoder[prefix_model.active_adapter].state_dict()
emb_params = sum(t.numel() for k, t in state.items() if "embedding" in k)
mlp_params = sum(t.numel() for t in state.values()) - emb_params

print(f"""
Prefix-tuning
  Frozen LM                      {frozen:>12,}
  Trainable (P' + MLP_theta)     {trainable:>12,}   {100*trainable/frozen:6.2f}% of LM
    of which embedding (P')      {emb_params:>12,}
    of which projection (MLP)    {mlp_params:>12,}

Full fine-tuning
  Trainable                      {frozen:>12,}   100.00% of LM
""")

### <font color="maroon"> **Future Direction**</font>
1. Play with predix projections disabled.
2. Play with other prefix sizes (e.g., 20,300,400...)
3. Play with bottleneck.

In [ ]:
# Thank you!